Testing How well YOLO model does on a video.

In [1]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
results = model.predict(
    r"C:\Users\rohan\Desktop\Quant Sports Project\Tester video\08fd33_4.mp4",
    save=True
)


Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\rohan\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.

WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/750) C:\Users\rohan\Desktop\Quant Sports Project\Tester video\08fd33_4.mp4: 384x640 22 persons, 1 sports ball, 101.5ms
video 1/1 (fram

## Findings (2026-09-18): YOLO26n, pretrained (COCO), inference only

Reviewed the annotated output on `08fd33_4.mp4`. Three problems observed with the stock pretrained detector:

1. **Ball detection is unreliable.** `sports ball` only fires intermittently and at low confidence (~0.3), missed in most frames, consistent with the known weakness of generic COCO-pretrained weights on small/fast objects. Will need to check whether this improves with a larger checkpoint (`yolo26x.pt`) or requires a soccer-ball-specific fine-tuned model.
2. **False positive class on broadcast graphics.** At least one frame classifies part of the on-screen broadcast graphic/scoreboard as `tv` (conf ~0.26). Low confidence, but worth filtering out before this feeds any downstream pipeline stage.
3. **Sideline personnel are detected as `person` indiscriminately.** Coaches, medical staff, and bench personnel outside the pitch are tracked with the same `person` class as players, no pitch-boundary or role filtering yet. This will need to be constrained (e.g. via homography/pitch-mask from Stage 2, or a jersey/role classifier) before player-only tracking is usable for factor construction.

Reference frame, showing all three issues at once (`tv` false positive top-left, faint `sports ball` detection right side, sideline personnel picked up as `person` at bottom):

![YOLO26n pretrained detection, tv false positive, weak ball detection, sideline personnel false positives](reference_frame_findings.png)

**Implication for next steps:** stock pretrained YOLO26 is not sufficient as-is for Stage 1 (detection & tracking). At minimum needs: (a) a pitch-boundary mask to exclude sideline detections, (b) evaluation of a larger/soccer-tuned checkpoint for ball detection, (c) confidence thresholding to drop low-confidence spurious classes like `tv`. None of this is a locked decision yet, just diagnostic input for how Stage 1 gets built.


## Decision (2026-09-18): retrain on a Roboflow dataset

Based on the findings above, stock pretrained YOLO26n is not usable as-is for Stage 1. Decision: fine-tune YOLO26 on **"Football Players Detection"** (Roboflow Universe) instead of relying on generic COCO weights.

**Why this dataset:** it covers all three problems surfaced in the smoke test in one source, rather than needing separate fixes:
- Player detection: soccer-specific, should reduce the class confusion seen with generic COCO weights.
- Ball detection: soccer-specific ball annotations, targets the unreliable/low-confidence `sports ball` detections.
- Referee / sideline personnel: a distinct class from players, which directly addresses the sideline-personnel false-positive finding without needing a separate pitch-mask step first.

**Licensing:** checked, permissive (CC BY 4.0). Keep a copy of the license terms on file alongside the DFL Kaggle confirmation, consistent with how licensing has been tracked for other data sources in this project.

**Status:** decision made, fine-tuning run not yet executed. This is a structural/pipeline decision (affects Stage 1: detection & tracking), so once `decision_log.md` exists at the project root it should be logged there too, not just in this notebook.


Get Data Set

In [1]:
import os
from dotenv import load_dotenv
from roboflow import Roboflow

load_dotenv()  # reads .env at project root (gitignored)

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-2frwp")
version = project.version(1)
dataset = version.download("yolo26")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Football-Players-Detection-1 in yolo26:: 100%|██████████| 2258/2258 [00:02<00:00, 799.71it/s]


In [ ]:
# NOTE: the original version of this cell moved train/test/valid into a nested
# Football-Players-Detection-1/Football-Players-Detection-1/ subfolder, which broke
# data.yaml's relative paths. Reverted manually; data.yaml now has an explicit
# absolute `path:` key instead, so no folder move is needed here. Left as a no-op
# to preserve the cell history.
pass


Training Model

In [ ]:
!yolo task=detect mode=train model=yolo26n.pt data="{dataset.location}/data.yaml" epochs=100 imgsz=640 batch=16
